# **Expected Gradients: baseline как распределение**

Практика к модулю [«Атрибуция от аксиом: IG, DeepLIFT, LRP»](https://open-xai-platform.web.app).

В уроке про выбор baseline мы пришли к неудобному выводу: **baseline — это и есть вопрос,
на который отвечает объяснение**, а у любой единственной точки есть слепое пятно. Чёрный
baseline теряет всё тёмное, белый — всё светлое.

Рекомендация урока — брать Expected Gradients, где baseline не точка, а распределение.
Здесь мы это проверим руками:

- увидим слепое пятно чёрного baseline **числом**, а не на словах;
- реализуем Expected Gradients в десять строк и убедимся, что пятно исчезает;
- проверим completeness обоих методов — сумма атрибуций против разности прогнозов.

In [ ]:
import torch
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0);

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load(name):
    return Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/' + name).content)).convert('RGB')

image = load('hog.jpg')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    target = model(x).argmax().item()
print('Класс:', target, categories[target])

## 1. Integrated Gradients с одним baseline

Формула из урока, приближённая суммой Римана:

$$\text{IG}_i(x) = (x_i - x'_i)\times\frac1m\sum_{k=1}^{m}
\frac{\partial F\big(x' + \tfrac{k}{m}(x-x')\big)}{\partial x_i}$$

Множитель $(x_i - x'_i)$ стоит перед интегралом — и именно он создаёт слепое пятно:
там, где вход совпал с baseline, атрибуция обнулится, каким бы ни был градиент.

In [ ]:
def grads_at(model, points, target):
    """Градиенты логита класса в каждой точке пути."""
    points = points.clone().requires_grad_(True)
    logits = model(points)[:, target].sum()
    g, = torch.autograd.grad(logits, points)
    return g

def integrated_gradients(model, x, baseline, target, m=32):
    alphas = torch.linspace(1 / m, 1.0, m).view(-1, 1, 1, 1)
    path = baseline + alphas * (x - baseline)
    g = grads_at(model, path, target).mean(0, keepdim=True)
    return (x - baseline) * g

# Осторожно: `torch.zeros_like(x)` — это НЕ чёрное изображение. Вход уже нормализован,
# и нули в нормализованных координатах соответствуют средней яркости ImageNet, то есть серому.
# Чёрный baseline получается прогоном чёрной картинки через тот же transform.
black = transform(Image.new('RGB', (224, 224), (0, 0, 0))).unsqueeze(0)
grey = torch.zeros_like(x)

ig_black = integrated_gradients(model, x, black, target)
ig_grey = integrated_gradients(model, x, grey, target)
print('чёрный baseline в нормализованных координатах:', [round(v, 2) for v in black[0, :, 0, 0].tolist()])
print('а нули соответствуют яркости:', [round(v, 2) for v in (grey[0, :, 0, 0] * 0 + 0.449).tolist()])

## 2. Слепое пятно — числом

Возьмём маску тёмных пикселей (в исходной шкале, до нормализации) и посмотрим, какая доля
всей «массы» атрибуции на них приходится. Если метод честно смотрит на тёмные области,
доля должна быть сопоставима с их площадью.

In [ ]:
raw = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])(image)
dark = (raw.mean(0) < 0.25)          # маска тёмных пикселей
print(f'тёмные пиксели занимают {dark.float().mean():.1%} площади')

def dark_share(attr):
    a = attr.abs().sum(1)[0]         # свернули каналы
    return (a[dark].sum() / a.sum()).item()

print(f'IG, чёрный baseline: на тёмное приходится {dark_share(ig_black):.1%} атрибуции')
print(f'IG, серый  baseline: на тёмное приходится {dark_share(ig_grey):.1%} атрибуции')

**Задание 1.** Постройте IG с **белым** baseline (`torch.ones_like(x)`) и посчитайте
ту же долю. Куда переехало слепое пятно?

In [ ]:
# Ваш код здесь

## 3. Expected Gradients

Baseline становится распределением: на каждом шаге берём случайный объект из набора данных
и случайную точку на пути к нему.

$$\text{EG}_i(x) = \mathbb{E}_{x'\sim D,\ \alpha\sim U(0,1)}
\Big[(x_i - x'_i)\cdot\frac{\partial F\big(x' + \alpha(x - x')\big)}{\partial x_i}\Big]$$

Обратите внимание, что это ровно то же выражение, что у IG, только математическое ожидание
берётся ещё и по baseline. Одна выборка на объект вместо $m$ шагов интегрирования.

In [ ]:
# «Датасет» baseline-ов: другие изображения курса. В своей задаче берите обучающую выборку.
pool = torch.cat([transform(load(n)).unsqueeze(0)
                  for n in ('cat.jpg', 'cat_and_dog.jpg', 'pig.png')])
print('объектов в пуле baseline-ов:', pool.shape[0])

def expected_gradients(model, x, pool, target, n=64):
    idx = torch.randint(0, pool.shape[0], (n,))
    baselines = pool[idx]
    alphas = torch.rand(n, 1, 1, 1)
    points = baselines + alphas * (x - baselines)
    g = grads_at(model, points, target)
    return ((x - baselines) * g).mean(0, keepdim=True)

eg = expected_gradients(model, x, pool, target)
print(f'Expected Gradients: на тёмное приходится {dark_share(eg):.1%} атрибуции')

**Задание 2.** Сравните три числа: долю площади тёмных пикселей, долю атрибуции у IG
с чёрным baseline и у Expected Gradients. Какое из двух объяснений теряет тёмные области?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(raw.permute(1, 2, 0)); ax[0].set_title('оригинал')
for a, (m, t) in zip(ax[1:], [(ig_black, 'IG, чёрный baseline'), (eg, 'Expected Gradients')]):
    a.imshow(m.abs().sum(1)[0].detach().numpy(), cmap='hot')
    a.set_title(t)
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

## 4. Completeness: сумма атрибуций против разности прогнозов

Аксиома полноты обещает $\sum_i \text{IG}_i(x) = F(x) - F(x')$. Проверить это — одна строка,
и именно так ловят слишком грубую аппроксимацию интеграла (в Captum эта невязка называется
convergence delta).

In [ ]:
with torch.no_grad():
    fx = model(x)[0, target].item()
    f_black = model(black)[0, target].item()
    f_pool = model(pool)[:, target].mean().item()

print(f'{"метод":28s} {"сумма атрибуций":>16s} {"F(x) − F(x′)":>14s} {"невязка":>10s}')
for name, attr, base in (('IG, чёрный baseline', ig_black, f_black),
                         ('Expected Gradients', eg, f_pool)):
    s = attr.sum().item()
    print(f'{name:28s} {s:16.3f} {fx - base:14.3f} {abs(s - (fx - base)):10.3f}')

**Задание 3.** Увеличьте число шагов IG с 32 до 128 и число выборок EG с 64 до 256.
Как меняется невязка у каждого метода? У какого она падает быстрее?

**Задание 4.** Совпадение с baseline обнуляет атрибуцию — проверьте это напрямую: занулите
в изображении квадрат 60×60 и посчитайте сумму модулей атрибуции внутри него для IG
с чёрным baseline. Ожидаемый ответ — ровно ноль.

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Слепое пятно измеримо.** Не «чёрный baseline теряет тёмное», а конкретная доля атрибуции,
  которую можно посчитать за три строки и положить в отчёт.
- **Expected Gradients — тот же IG**, у которого ожидание берётся ещё и по baseline. Кода
  почти столько же, а слепого пятна одной точки нет.
- **Completeness проверяется всегда.** Заметная невязка означает, что шагов мало и интеграл
  не сошёлся, — а не что метод плох.
- **Baseline указывается рядом с картинкой.** Карта без указания baseline информативна
  примерно как доверительный интервал без уровня доверия.